# Simple patient-level models on ResNet median embeddings

Each patient is summarized by the median of their time point 0/1 cell ResNet-18 embeddings,
and classified with leave-one-plate-out logistic regression and random forests. This is a
simpler, patient-level alternative to aggregating cell-level predictions by majority vote.

The result CSVs written here are consumed by the `Fig. S3` cells in
[`../figure_notebooks/figures_3_S3.ipynb`](../figure_notebooks/figures_3_S3.ipynb).

In [13]:
from data import PlateDataset

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision.models import resnet18 as make_resnet18
from torchvision.models.feature_extraction import create_feature_extractor

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


device = 'cuda:0'

In [14]:
data = PlateDataset([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])

100%|██████████| 16/16 [00:03<00:00,  4.90it/s]


In [15]:
def extract_resnet_patch_features(imgs, transform=None):
  model = make_resnet18(weights="DEFAULT").to(device)
  return_nodes = {
      'flatten': 'z',
  }
  feature_extractor = create_feature_extractor(model.eval().to(device), return_nodes=return_nodes)
  z = torch.zeros((len(imgs), 512))
  i = 0
  loader = DataLoader(imgs, batch_size=128, shuffle=False)
  for img_batch in tqdm(loader):
    img_batch = img_batch.to(device).repeat(1, 3, 1, 1)
    if transform is not None:
      img_batch = transform(img_batch)
    with torch.no_grad():
      z[i:i+len(img_batch)] = feature_extractor(img_batch)['z'].cpu()
    i += len(img_batch)
  return z


res_zs = extract_resnet_patch_features(data.imgs)
res_zs.shape

100%|██████████| 8547/8547 [02:58<00:00, 47.80it/s]


torch.Size([1093966, 512])

## Patient median embeddings

Each patient is represented by the **median** of their ResNet features over all of their time point
0 and 1 cells (same `> 100` cells per time point inclusion as the cell-level classifiers). Swap the
`.median()` below for a bag-subsampled median if you want to match the cell-level sampling instead.

In [16]:
def patient_median_embeddings(use_plates, use_groups=None, use_times=[0, 1]):
  # patients with > 100 cells at time point 0 or 1, as in the cell-level classifiers
  cdf = data.info[data.info['plate'].isin(use_plates)]
  if use_groups is not None:
    cdf = cdf[cdf['group'].isin(use_groups)]
  cdf = cdf.groupby(['patient', 'time'])['cell'].count().reset_index()
  cdf = cdf[cdf['cell'] > 100]
  p01s = cdf[cdf['time'].isin([0, 1])]['patient'].unique()

  use_idx = np.argwhere((data.info['plate'].isin(use_plates) & data.info['time'].isin(use_times)
                         & data.info['patient'].isin(p01s)).values).flatten()
  info = data.info.iloc[use_idx]
  emb = pd.DataFrame(res_zs.numpy()[use_idx])
  emb['plate'] = info['plate'].values
  emb['pat'] = info['patient'].values
  emb['group'] = info['group'].values
  # one median embedding per patient per plate
  return emb.groupby(['plate', 'pat', 'group']).median().reset_index()

## Leave-one-plate-out classification

Train on all plates but one, predict the held-out plate's patients, and concatenate the predictions
(the repo's standard cross-validation). Logistic regression is fit on standardized features.

In [17]:
def leave_one_plate_out(med, classifier):
  feat_cols = [c for c in med.columns if isinstance(c, int)]
  dfs = []
  for plate in tqdm(sorted(med['plate'].unique())):
    test = med[med['plate'] == plate]
    # also exclude the held-out plate's patients from training (they can also appear on other plates)
    train = med[(med['plate'] != plate) & (~med['pat'].isin(test['pat']))]
    clf = clone(classifier).fit(train[feat_cols].values, train['lab'].values)
    out = test[['plate', 'pat', 'group', 'lab']].copy()
    out['pred'] = clf.predict(test[feat_cols].values)
    dfs.append(out)
  return pd.concat(dfs, ignore_index=True)


logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf = RandomForestClassifier(n_estimators=500, random_state=0)

## Healthy vs. cancer (patient level)

Binary classifiers on the patient median embeddings, leave-one-plate-out over plates 1-16
(`lab`: healthy = 1, cancer = 0).

In [18]:
use_plates = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])
med = patient_median_embeddings(use_plates)
med['lab'] = (med['group'] == 'healthy').astype(int)  # healthy = 1, cancer = 0

leave_one_plate_out(med, logreg).to_csv(
    'results/1_16_t01_healthy_cancer_patient_median_resnet_logreg_leave_one_plate_out.csv', index=False)
leave_one_plate_out(med, rf).to_csv(
    'results/1_16_t01_healthy_cancer_patient_median_resnet_rf_leave_one_plate_out.csv', index=False)

100%|██████████| 16/16 [00:10<00:00,  1.56it/s]


## Healthy vs. main cancer groups (patient level)

Multiclass (healthy / H&N / Meningioma / Chordoma-Chondrosarcoma) classifiers on the patient median
embeddings, leave-one-plate-out over plates 3-12.

In [19]:
group_map = {'healthy': 0, 'H&N cancer': 1, 'CNS-Meningioma': 2, 'Chordoma/Chondrosarcoma': 3}
use_groups = list(group_map)
use_plates = np.array([3, 4, 5, 6, 7, 8, 9, 10, 11, 12])

med = patient_median_embeddings(use_plates, use_groups=use_groups)
med['lab'] = med['group'].map(group_map)

leave_one_plate_out(med, logreg).to_csv(
    'results/3_12_t01_healthy_groups_patient_median_resnet_logreg_leave_one_plate_out.csv', index=False)
leave_one_plate_out(med, rf).to_csv(
    'results/3_12_t01_healthy_groups_patient_median_resnet_rf_leave_one_plate_out.csv', index=False)

100%|██████████| 10/10 [00:06<00:00,  1.56it/s]
